# tp-05b モデル族の比較（LightGBM・CatBoost・XGBoost・線形・NN）

**実行場所: ホスト PC（stg）。** tp-05a の最良の特徴量（B0′ + 履歴 127 列 + 前段 3 列 + 相対化）で、モデル族を同じ条件で比べる。
各モデルは p_win・p_top2・p_top3 を出し、レース内で合計 1・2・3 にそろえる。softmax 系で p_top2・p_top3 を直接出さないものは、割引 Harville（OOF で当てはめた γ）で導く。
**OOF（検証 fold の予測）を保存**し、tp-05e のアンサンブルが使う。評価は p_win・p_top2・p_top3 の LogLoss と AUC・Brier・ECE・1 位的中率、MLflow（`keiba-tp-model-zoo`）に記録する。

| モデル | 内容 |
|---|---|
| lgb_binary | LightGBM 二値 × 3（基準） |
| lgb_softmax | LightGBM レース内 softmax（p_top2・p_top3 は Harville） |
| cat_binary | CatBoost 二値 × 3（GPU、カテゴリ列をそのまま使う） |
| xgb_binary | XGBoost 二値 × 3（GPU） |
| linear | 線形（レース内 softmax + top2・top3 の BCE、GPU） |
| nn_s / nn / nn_l | レース内 NN（馬ごとの MLP + 同レース平均の文脈、listwise softmax + BCE）。隠れ層 (128,64) / (256,128) / (512,256,128) |

条件: valid 2022・2023・2024（expanding、purge 7 日 + embargo 3 日）、3 seed、2026 年と test 2025 は使わない。ホストのメモリと 6 コアを空ける。

In [ ]:
import os, sys
from pathlib import Path


def _find_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "AGENTS.md").exists() and (q / "src").is_dir():
            return q
    raise RuntimeError("リポジトリルートが見つかりません")


REPO_ROOT = _find_root()
os.chdir(REPO_ROOT)
os.environ.setdefault("POLARS_MAX_THREADS", "6")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.utils.project_env import load_project_dotenv
load_project_dotenv()
print("REPO_ROOT:", REPO_ROOT, "| KEIBA_ENV:", os.environ.get("KEIBA_ENV"))

In [ ]:
import json, subprocess, time, gc
import numpy as np
import pandas as pd
import mlflow
import psutil

from src.pipeline.mlflow.train_utils import require_tracking_server
from src.pipeline.models import tp05_data, zoo_runner
from src.pipeline.models.compute import detect_compute

WORK = REPO_ROOT / "data/local/modeling/training_pipeline"
OOF_DIR = WORK / "oof/model_oof"
OUT = WORK / "model_zoo"
OUT.mkdir(parents=True, exist_ok=True)
THREADS = detect_compute(reserve_cores=6).threads
SEEDS = [0, 1, 2]
mlflow.set_experiment("keiba-tp-model-zoo")
require_tracking_server()
GIT_COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, cwd=REPO_ROOT).stdout.strip()
PROC = psutil.Process()


def mem(label=""):
    vm = psutil.virtual_memory()
    print(f"[メモリ] {label}: この process {PROC.memory_info().rss / 1e9:.1f}GB / ホスト使用 {vm.used / 1e9:.0f}GB ({vm.percent:.0f}%)")
    return PROC.memory_info().rss / 1e9, vm.percent


D = tp05_data.load_dataset(WORK)
df, meta, folds = D["df"], D["meta"], D["folds"]
X = df[D["FEATS"]["base+S+C1"]]
print(f"行 {len(df):,} / 特徴量 {X.shape[1]} 列 / fold {[(f.fold_id, len(f.train_index), len(f.valid_index)) for f in folds]} / threads {THREADS}")
mem("読み込み後")

In [ ]:
MODELS = {  # 名前: (recipe, params)
    "lgb_binary": ("lgb_binary", {}),
    "lgb_softmax": ("lgb_softmax", {}),
    "cat_binary": ("cat_binary", {"task_type": "GPU"}),
    "xgb_binary": ("xgb_binary", {"device": "cuda"}),
    "linear": ("linear", {"epochs": 14}),
    "nn_s": ("nn", {"hidden": (128, 64), "epochs": 14}),
    "nn": ("nn", {"hidden": (256, 128), "epochs": 14}),
    "nn_l": ("nn", {"hidden": (512, 256, 128), "epochs": 14, "dropout": 0.2}),
}
RES = []
for name, (recipe, params) in MODELS.items():
    with mlflow.start_run(run_name=f"tp-05b-{name}", tags={"notebook": "tp-05b", "git_commit": GIT_COMMIT, "model": name, "recipe": recipe, "cv": "expanding valid 2022-2024"}):
        try:
            res, oof = zoo_runner.run_oof(name, recipe, X, meta, folds, SEEDS, THREADS, params=params, out_dir=OOF_DIR)
        except Exception as e:      # GPU が使えない等。CPU にフォールバックして記録に残す
            print(f"  {name}: 失敗 {type(e).__name__}: {str(e)[:120]} → CPU で再試行")
            p2 = dict(params); p2["task_type"] = "CPU"; p2["device"] = "cpu"
            res, oof = zoo_runner.run_oof(name, recipe, X, meta, folds, SEEDS, THREADS, params=p2, out_dir=OOF_DIR)
            mlflow.set_tag("fallback", "cpu")
        rss, pct = mem(name)
        num = [c for c in res.columns if c not in ("model", "fold")]
        mlflow.log_params({"recipe": recipe, "n_features": X.shape[1], "seeds": str(SEEDS), **{f"p_{k}": str(v) for k, v in params.items()}})
        mlflow.log_metrics({f"mean_{c}": float(res[c].mean()) for c in num})
        for _, r in res.iterrows():
            mlflow.log_metrics({f"{r['fold']}_{c}": float(r[c]) for c in ("logloss_win", "logloss_top2", "logloss_top3", "race_nll")})
        mlflow.log_metrics({"seconds": res.attrs["seconds"], "rss_gb": rss, "host_mem_percent": pct})
        res["seconds"] = res.attrs["seconds"]
        RES.append(res)
        print(f"  {name:12s} win {res['logloss_win'].mean():.5f}  top2 {res['logloss_top2'].mean():.5f}  top3 {res['logloss_top3'].mean():.5f}  ({res.attrs['seconds']:.0f}s)")
    pd.concat(RES).to_csv(OUT / "fold_metrics.csv", index=False)
    gc.collect()

In [ ]:
R = pd.concat(RES, ignore_index=True)
cols = ["logloss_win", "logloss_top2", "logloss_top3", "race_nll", "brier_win", "brier_top2", "brier_top3", "ece_win", "auc_win", "auc_top2", "auc_top3", "top1_hit", "ndcg3"]
summary = R.groupby("model")[cols].mean().loc[list(MODELS)]
summary["seconds"] = R.groupby("model")["seconds"].first().loc[list(MODELS)]
display(summary.round(5))
pf = R.pivot(index="model", columns="fold", values="logloss_win").loc[list(MODELS)]
display(pf.round(5))
summary.to_csv(OUT / "summary.csv"); pf.to_csv(OUT / "per_fold_logloss_win.csv")
with mlflow.start_run(run_name="tp-05b-summary", tags={"notebook": "tp-05b", "git_commit": GIT_COMMIT}):
    for m in summary.index:
        for c in ("logloss_win", "logloss_top2", "logloss_top3"):
            mlflow.log_metric(f"{c}__{m}", float(summary.loc[m, c]))
    mlflow.log_artifact(str(OUT / "summary.csv"), "model_zoo")
mem("終了")